# Notebook 2 — GhostNeck-YOLOv8n (5-Fold Cross Validation)

**Project:** Lightweight Fall Detection for Elderly Care Using GhostNeck-YOLOv8 on Edge Devices  
**Proposed model:** GhostNeck-YOLOv8n — replaces all C2f modules in the YOLOv8n Neck with C3Ghost blocks.  
**Validation:** 5-Fold Stratified Cross Validation (same folds as Baseline for fair comparison)  
**Environment:** Google Colab (Tesla T4 GPU)

This notebook uses **identical fold splits, seed, and training config** as the Baseline notebook to ensure a fair comparison.

---
## Step 1 — Environment Setup

In [ ]:
!pip install ultralytics scikit-learn -q

import torch
print(f"PyTorch : {torch.__version__}")
print(f"CUDA    : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU     : {torch.cuda.get_device_name(0)}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 36.2 MB/s eta 0:00:00
PyTorch : 2.10.0+cu128
CUDA    : True
GPU     : Tesla T4


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print("Google Drive mounted.")

Mounted at /content/drive
Google Drive mounted.


---
## Step 2 — Extract Dataset and Merge into Pool

In [ ]:
import os, zipfile, shutil, yaml
from pathlib import Path
from collections import Counter

# ============================================================
# CONFIG — must match Baseline notebook exactly
# ============================================================
DRIVE_ZIP   = "/content/drive/MyDrive/fall_detection_clean.zip"
EXTRACT_DIR = "/content/dataset_raw"
POOL_DIR    = "/content/dataset_pool"
OUTPUT_DIR  = "/content/runs"
K_FOLDS     = 5
SEED        = 42        # MUST match Baseline to get identical folds
# ============================================================

if not os.path.exists(DRIVE_ZIP):
    alt = DRIVE_ZIP.replace(".zip", "")
    if os.path.exists(alt):
        DRIVE_ZIP = alt
    else:
        raise FileNotFoundError(f"Dataset not found: {DRIVE_ZIP}")

if os.path.exists(EXTRACT_DIR):
    shutil.rmtree(EXTRACT_DIR)
with zipfile.ZipFile(DRIVE_ZIP, 'r') as zf:
    zf.extractall(EXTRACT_DIR)
print(f"Extracted to {EXTRACT_DIR}")

yaml_files = list(Path(EXTRACT_DIR).rglob("data.yaml"))
dataset_root = yaml_files[0].parent
print(f"Dataset root: {dataset_root}")

# Merge all splits into one pool
pool_img = Path(POOL_DIR) / "images"
pool_lbl = Path(POOL_DIR) / "labels"
if os.path.exists(POOL_DIR):
    shutil.rmtree(POOL_DIR)
pool_img.mkdir(parents=True)
pool_lbl.mkdir(parents=True)

img_count = 0
for split in ["train", "valid", "test", "val"]:
    img_dir = dataset_root / split / "images"
    lbl_dir = dataset_root / split / "labels"
    if not img_dir.exists():
        continue
    for img_path in img_dir.glob("*"):
        if img_path.suffix.lower() in [".jpg", ".jpeg", ".png"]:
            shutil.copy2(str(img_path), str(pool_img / img_path.name))
            lbl_path = lbl_dir / (img_path.stem + ".txt")
            if lbl_path.exists():
                shutil.copy2(str(lbl_path), str(pool_lbl / lbl_path.name))
            img_count += 1

print(f"Merged pool: {img_count} images")

Extracted to /content/dataset_raw
Dataset root: /content/dataset_raw/fall_detection_clean
Merged pool: 474 images


---
## Step 3 — Generate Identical 5-Fold Splits

In [ ]:
import numpy as np
from sklearn.model_selection import StratifiedKFold

all_images = sorted(list(pool_img.glob("*.jpg")) + list(pool_img.glob("*.png")))
image_names = []
image_classes = []

for img_path in all_images:
    lbl_path = pool_lbl / (img_path.stem + ".txt")
    if lbl_path.exists():
        text = lbl_path.read_text().strip()
        if text:
            primary_cls = int(text.splitlines()[0].split()[0])
            stratify_cls = 0 if primary_cls == 0 else 1
            image_names.append(img_path.name)
            image_classes.append(stratify_cls)

image_names = np.array(image_names)
image_classes = np.array(image_classes)

# Same seed = same folds as Baseline
skf = StratifiedKFold(n_splits=K_FOLDS, shuffle=True, random_state=SEED)
folds = list(skf.split(image_names, image_classes))

print(f"Total images: {len(image_names)}")
print(f"{K_FOLDS}-Fold split (identical to Baseline):")
for i, (train_idx, val_idx) in enumerate(folds):
    train_cls = Counter(image_classes[train_idx])
    val_cls   = Counter(image_classes[val_idx])
    print(f"  Fold {i+1}: train={len(train_idx)} (fall={train_cls[0]}, nf={train_cls[1]})  |  "
          f"val={len(val_idx)} (fall={val_cls[0]}, nf={val_cls[1]})")

Total images: 474
5-Fold split (identical to Baseline):
  Fold 1: train=379 (fall=221, nf=158)  |  val=95 (fall=56, nf=39)
  Fold 2: train=379 (fall=221, nf=158)  |  val=95 (fall=56, nf=39)
  Fold 3: train=379 (fall=222, nf=157)  |  val=95 (fall=55, nf=40)
  Fold 4: train=379 (fall=222, nf=157)  |  val=95 (fall=55, nf=40)
  Fold 5: train=380 (fall=222, nf=158)  |  val=94 (fall=55, nf=39)


---
## Step 4 — Define GhostNeck Architecture

In [ ]:
ghost_yaml_content = """
# GhostNeck-YOLOv8n
# Backbone: unchanged YOLOv8n (loads COCO pretrained weights)
# Neck: C2f replaced by C3Ghost; strided Conv replaced by GhostConv
# Reference: Han et al., GhostNet, CVPR 2020

nc: 80

scales:
  n: [0.33, 0.25, 1024]

backbone:
  - [-1, 1, Conv,  [64, 3, 2]]
  - [-1, 1, Conv,  [128, 3, 2]]
  - [-1, 3, C2f,   [128, True]]
  - [-1, 1, Conv,  [256, 3, 2]]
  - [-1, 6, C2f,   [256, True]]
  - [-1, 1, Conv,  [512, 3, 2]]
  - [-1, 6, C2f,   [512, True]]
  - [-1, 1, Conv,  [1024, 3, 2]]
  - [-1, 3, C2f,   [1024, True]]
  - [-1, 1, SPPF,  [1024, 5]]

head:
  - [-1, 1, nn.Upsample, [None, 2, 'nearest']]
  - [[-1, 6], 1, Concat, [1]]
  - [-1, 3, C3Ghost, [512]]
  - [-1, 1, nn.Upsample, [None, 2, 'nearest']]
  - [[-1, 4], 1, Concat, [1]]
  - [-1, 3, C3Ghost, [256]]
  - [-1, 1, GhostConv, [256, 3, 2]]
  - [[-1, 12], 1, Concat, [1]]
  - [-1, 3, C3Ghost, [512]]
  - [-1, 1, GhostConv, [512, 3, 2]]
  - [[-1, 9], 1, Concat, [1]]
  - [-1, 3, C3Ghost, [1024]]
  - [[15, 18, 21], 1, Detect, [nc]]
"""

GHOST_YAML = "/content/yolov8n-ghostneck.yaml"
with open(GHOST_YAML, 'w') as f:
    f.write(ghost_yaml_content)

# Verify parameter count
from ultralytics import YOLO
ghost_test = YOLO(GHOST_YAML)
g_params = sum(p.numel() for p in ghost_test.model.parameters())

baseline_test = YOLO("yolov8n.yaml")
b_params = sum(p.numel() for p in baseline_test.model.parameters())

reduction = (1 - g_params / b_params) * 100
print(f"Baseline : {b_params/1e6:.2f}M")
print(f"GhostNeck: {g_params/1e6:.2f}M")
print(f"Reduction: {reduction:.1f}%")

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Baseline : 3.16M
GhostNeck: 2.57M
Reduction: 18.5%


---
## Step 5 — Train GhostNeck on Each Fold

Uses identical fold splits, training config, and augmentation as Baseline.

In [ ]:
EPOCHS     = 100
BATCH_SIZE = 16
IMG_SIZE   = 640

all_fold_metrics = []

for fold_idx, (train_idx, val_idx) in enumerate(folds):
    fold_num = fold_idx + 1
    print("\n" + "=" * 65)
    print(f"  FOLD {fold_num}/{K_FOLDS}  —  train: {len(train_idx)}  |  val: {len(val_idx)}")
    print("=" * 65)

    # --- Create fold-specific dataset directory ---
    fold_dir = Path(f"/content/fold_{fold_num}")
    if fold_dir.exists():
        shutil.rmtree(fold_dir)

    for split, indices in [("train", train_idx), ("val", val_idx)]:
        (fold_dir / split / "images").mkdir(parents=True)
        (fold_dir / split / "labels").mkdir(parents=True)
        for idx in indices:
            name = image_names[idx]
            stem = Path(name).stem
            shutil.copy2(str(pool_img / name),
                         str(fold_dir / split / "images" / name))
            lbl_name = stem + ".txt"
            if (pool_lbl / lbl_name).exists():
                shutil.copy2(str(pool_lbl / lbl_name),
                             str(fold_dir / split / "labels" / lbl_name))

    # --- Write fold-specific data.yaml ---
    fold_yaml = {
        "path":  str(fold_dir),
        "train": "train/images",
        "val":   "val/images",
        "nc":    3,
        "names": ["fall_detected", "sitting", "walking"],
    }
    fold_yaml_path = str(fold_dir / "data.yaml")
    with open(fold_yaml_path, 'w') as f:
        yaml.dump(fold_yaml, f, default_flow_style=False)

    # --- Train GhostNeck ---
    model = YOLO(GHOST_YAML).load("yolov8n.pt")
    results = model.train(
        data    = fold_yaml_path,
        epochs  = EPOCHS,
        imgsz   = IMG_SIZE,
        batch   = BATCH_SIZE,
        project = OUTPUT_DIR,
        name    = f"ghostneck_fold{fold_num}",
        optimizer      = "SGD",
        lr0            = 0.01,
        lrf            = 0.01,
        momentum       = 0.937,
        weight_decay   = 0.0005,
        warmup_epochs  = 3.0,
        warmup_momentum= 0.8,
        mosaic       = 1.0,
        mixup        = 0.1,
        close_mosaic = 10,
        fliplr       = 0.5,
        flipud       = 0.0,
        degrees      = 5.0,
        translate    = 0.1,
        scale        = 0.5,
        shear        = 2.0,
        perspective  = 0.0,
        hsv_h        = 0.015,
        hsv_s        = 0.7,
        hsv_v        = 0.4,
        erasing      = 0.0,
        copy_paste   = 0.0,
        patience = 20,
        seed     = SEED,
        workers  = 2,
        exist_ok = True,
        verbose  = True,
    )

    # --- Evaluate ---
    best_pt = Path(OUTPUT_DIR) / f"ghostneck_fold{fold_num}" / "weights" / "best.pt"
    eval_model = YOLO(str(best_pt))
    metrics = eval_model.val(data=fold_yaml_path)

    model_size = best_pt.stat().st_size / (1024**2)
    param_count = sum(p.numel() for p in eval_model.model.parameters()) / 1e6

    fall_ap50 = float(metrics.box.ap50[0]) if len(metrics.box.ap50) > 0 else 0.0
    fall_ap   = float(metrics.box.ap[0])   if len(metrics.box.ap) > 0   else 0.0

    fold_result = {
        "fold":       fold_num,
        "map50":      metrics.box.map50,
        "map50_95":   metrics.box.map,
        "precision":  metrics.box.mp,
        "recall":     metrics.box.mr,
        "fall_ap50":  fall_ap50,
        "fall_ap":    fall_ap,
        "params_m":   param_count,
        "size_mb":    model_size,
    }
    all_fold_metrics.append(fold_result)

    print(f"\n  Fold {fold_num} results:")
    print(f"    mAP@0.5        : {metrics.box.map50:.4f}")
    print(f"    mAP@0.5:0.95   : {metrics.box.map:.4f}")
    print(f"    Precision      : {metrics.box.mp:.4f}")
    print(f"    Recall         : {metrics.box.mr:.4f}")
    print(f"    fall_detected AP@0.5 : {fall_ap50:.4f}")

    shutil.rmtree(fold_dir)

print("\n" + "=" * 65)
print("All folds complete.")
print("=" * 65)


  FOLD 1/5  —  train: 379  |  val: 95
Transferred 255/439 items from pretrained weights
Ultralytics 8.4.43 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/fold_1/data.yaml, degrees=5.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.0, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.1, mode=train, model=/content/yolov8n-ghostneck.yaml, momentum=0.937, mosaic=1.0, multi_scale=0.0, n

---
## Step 6 — Results Summary (Mean ± Std)

In [ ]:
import statistics

print("\n" + "=" * 70)
print("GhostNeck-YOLOv8n — 5-Fold Cross Validation Results")
print("=" * 70)

# Per-fold table
print(f"\n{'Fold':<6} {'mAP@0.5':>10} {'mAP@50:95':>12} {'Precision':>12} {'Recall':>10} {'fall AP@0.5':>13}")
print("-" * 65)
for r in all_fold_metrics:
    print(f"  {r['fold']:<4} {r['map50']:>10.4f} {r['map50_95']:>12.4f} {r['precision']:>12.4f} {r['recall']:>10.4f} {r['fall_ap50']:>13.4f}")

# Mean ± Std
print("-" * 65)
metrics_keys = {
    'mAP@0.5':        'map50',
    'mAP@0.5:0.95':   'map50_95',
    'Precision':       'precision',
    'Recall':          'recall',
    'fall AP@0.5':     'fall_ap50',
    'fall AP@0.5:0.95':'fall_ap',
}

summary = {}
print(f"\n{'Metric':<20} {'Mean':>10} {'± Std':>10}")
print("-" * 42)
for display_name, key in metrics_keys.items():
    values = [r[key] for r in all_fold_metrics]
    mean = statistics.mean(values)
    std  = statistics.stdev(values) if len(values) > 1 else 0.0
    summary[key] = {'mean': mean, 'std': std}
    print(f"  {display_name:<18} {mean:>10.4f} {'±':>3} {std:<8.4f}")

print(f"\n  Parameters : {all_fold_metrics[0]['params_m']:.2f}M")
print(f"  Model size : {all_fold_metrics[0]['size_mb']:.2f} MB")


GhostNeck-YOLOv8n — 5-Fold Cross Validation Results

Fold      mAP@0.5    mAP@50:95    Precision     Recall   fall AP@0.5
-----------------------------------------------------------------
  1        0.8107       0.5417       0.8436     0.7709        0.9782
  2        0.8713       0.5648       0.8686     0.8163        0.9759
  3        0.8760       0.5894       0.9248     0.7741        0.9511
  4        0.6970       0.4438       0.7589     0.6683        0.9466
  5        0.9122       0.6092       0.8571     0.8485        0.9665
-----------------------------------------------------------------

Metric                     Mean      ± Std
------------------------------------------
  mAP@0.5                0.8334   ± 0.0845  
  mAP@0.5:0.95           0.5498   ± 0.0645  
  Precision              0.8506   ± 0.0599  
  Recall                 0.7756   ± 0.0680  
  fall AP@0.5            0.9637   ± 0.0143  
  fall AP@0.5:0.95       0.6609   ± 0.0468  

  Parameters : 2.42M
  Model size : 4.89 M

---
## Step 7 — Save Results + Export Best Model

In [ ]:
import json

SAVE_DIR = Path("/content/drive/MyDrive/fall_detection_kfold_results/ghostneck")
SAVE_DIR.mkdir(parents=True, exist_ok=True)

results_dict = {
    "model":       "GhostNeck-YOLOv8n",
    "k_folds":     K_FOLDS,
    "seed":        SEED,
    "epochs":      EPOCHS,
    "total_images": len(image_names),
    "params_m":    all_fold_metrics[0]['params_m'],
    "size_mb":     all_fold_metrics[0]['size_mb'],
    "per_fold":    all_fold_metrics,
    "summary":     {k: {"mean": round(v["mean"], 4), "std": round(v["std"], 4)}
                    for k, v in summary.items()},
}

with open(SAVE_DIR / "ghostneck_kfold_results.json", 'w') as f:
    json.dump(results_dict, f, indent=2)

# Save best fold's weights
best_fold = max(all_fold_metrics, key=lambda x: x['fall_ap50'])
best_fold_pt = Path(OUTPUT_DIR) / f"ghostneck_fold{best_fold['fold']}" / "weights" / "best.pt"
if best_fold_pt.exists():
    shutil.copy2(str(best_fold_pt), str(SAVE_DIR / "best.pt"))
    print(f"Best fold: Fold {best_fold['fold']} (fall AP@0.5 = {best_fold['fall_ap50']:.4f})")

# Export best model to ONNX FP32 and FP16
print("\nExporting best model ...")
best_model = YOLO(str(SAVE_DIR / "best.pt"))

onnx_path = best_model.export(format="onnx", imgsz=640)
onnx_size = os.path.getsize(onnx_path) / 1024**2
shutil.copy2(onnx_path, str(SAVE_DIR / "best_32.onnx"))
print(f"  ONNX FP32: {onnx_size:.2f} MB")

fp16_path = best_model.export(format="onnx", imgsz=640, half=True)
fp16_size = os.path.getsize(fp16_path) / 1024**2
shutil.copy2(fp16_path, str(SAVE_DIR / "best_fp16.onnx"))
print(f"  ONNX FP16: {fp16_size:.2f} MB")

print(f"\nAll results saved to: {SAVE_DIR}")
print(json.dumps(results_dict['summary'], indent=2))

Best fold: Fold 1 (fall AP@0.5 = 0.9782)

Exporting best model ...
Ultralytics 8.4.43 🚀 Python-3.12.13 torch-2.10.0+cu128 CPU (Intel Xeon CPU @ 2.00GHz)
YOLOv8n-ghostneck summary (fused): 103 layers, 2,422,529 parameters, 0 gradients, 7.0 GFLOPs

PyTorch: starting from '/content/drive/MyDrive/fall_detection_kfold_results/ghostneck/best.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 7, 8400) (4.9 MB)

ONNX: starting export with onnx 1.21.0 opset 20...
ONNX: slimming with onnxslim 0.1.92...
ONNX: export success ✅ 3.7s, saved as '/content/drive/MyDrive/fall_detection_kfold_results/ghostneck/best.onnx' (9.5 MB)

Export complete (4.3s)
Results saved to /content/drive/MyDrive/fall_detection_kfold_results/ghostneck
Predict:         yolo predict task=detect model=/content/drive/MyDrive/fall_detection_kfold_results/ghostneck/best.onnx imgsz=640 
Validate:        yolo val task=detect model=/content/drive/MyDrive/fall_detection_kfold_results/ghostneck/best.onnx imgsz=640 data=

In [ ]:
fp16_path = best_model.export(format="onnx", imgsz=640, half=True)
print(f"ONNX FP16: {os.path.getsize(fp16_path)/1024**2:.2f} MB")

Ultralytics 8.4.43 🚀 Python-3.12.13 torch-2.10.0+cu128 CPU (Intel Xeon CPU @ 2.00GHz)
YOLOv8n-ghostneck summary (fused): 103 layers, 2,422,529 parameters, 0 gradients, 7.0 GFLOPs

PyTorch: starting from '/content/drive/MyDrive/fall_detection_kfold_results/ghostneck/best.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 7, 8400) (4.9 MB)

ONNX: starting export with onnx 1.21.0 opset 20...
ONNX: slimming with onnxslim 0.1.92...
ONNX: converting to FP16...
ONNX: export success ✅ 1.5s, saved as '/content/drive/MyDrive/fall_detection_kfold_results/ghostneck/best.onnx' (4.8 MB)

Export complete (2.3s)
Results saved to /content/drive/MyDrive/fall_detection_kfold_results/ghostneck
Predict:         yolo predict task=detect model=/content/drive/MyDrive/fall_detection_kfold_results/ghostneck/best.onnx imgsz=640 half
Validate:        yolo val task=detect model=/content/drive/MyDrive/fall_detection_kfold_results/ghostneck/best.onnx imgsz=640 data=/content/fold_1/data.yaml half 
Vis